In [6]:
import csv

csv_path = '/home/ws-ids-es3-01/PycharmProjects/hamdard_bm2cp/opencood/logs/attentioncomm.csv'

total_first = 0
total_second = 0

with open(csv_path, 'r') as f:
    reader = csv.reader(f)
    next(reader)  # Skip header
    for row in reader:
        values = eval(row[1])
        total_first += values[0]
        total_second += values[1]

ratio = total_first / total_second if total_second != 0 else 0
import math

comm_value = math.log2(total_first * 32 / 8) if total_first != 0 else 0
print(f"Communication value (log2 scale): {comm_value:.3f}")
print(f"Total first: {total_first}")
print(f"Total second: {total_second}")
print(f"Ratio (first/second): {ratio:.6f}")

Communication value (log2 scale): 33.053
Total first: 2228438205
Total second: 12024004608
Ratio (first/second): 0.185332


In [7]:
import pandas as pd
import glob
import os
import ast
# --- 1. CONFIGURE YOUR LOGS DIRECTORY ---
# Set the path to the single folder containing all your model CSV files.
log_directory = '/home/ws-ids-es3-01/PycharmProjects/hamdard_bm2cp/opencood/logs/BM2CP_MLP_RCA_Bachelorthesisversion/adver_city_bm2cp_radar_camera_2025_05_22_01_41_18_gleiche/bandwidth'

# --- 2. FIND AND PROCESS EACH MODEL'S CSV FILE ---
csv_files = glob.glob(os.path.join(log_directory, '*.csv'))
results_list = []


In [8]:
if not csv_files:
    print(f"⚠️ Error: No CSV files were found in '{log_directory}'")
else:
    for file_path in csv_files:
        model_name = os.path.splitext(os.path.basename(file_path))[0]

        try:
            df = pd.read_csv(file_path)

            # --- NEW: PARSE THE 'Value' COLUMN ---
            # Safely evaluate the string representation of the dictionary
            # The lambda function handles potential errors if a value is not a valid string literal
            parsed_data = df['Value'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else {})

            # Expand the parsed dictionaries into separate columns
            expanded_df = pd.json_normalize(parsed_data)

            # --- Calculate Averages from the new columns ---
            avg_comm = expanded_df['Comm'].mean() if 'Comm' in expanded_df.columns else None

            # The new format provides 'total_mb'. We will average this value.
            avg_mb = expanded_df['total_mb'].mean() if 'total_mb' in expanded_df.columns else None

            # Append results to our list
            results_list.append({
                'Model': model_name,
                'Average Comm': avg_comm,
                'Average Total MB': avg_mb  # Changed column name for clarity
            })
        except Exception as e:
            print(f"Could not process file {model_name}.csv due to an error: {e}")


# --- 3. CREATE AND DISPLAY THE FINAL SUMMARY TABLE ---
if results_list:
    summary_df = pd.DataFrame(results_list)
    summary_df = summary_df.set_index('Model')
    summary_df.sort_index(inplace=True)

    print("--- Bandwidth Metrics Summary ---")

    # Use the corrected to_string() method with formatters
    print(summary_df.to_string(
        formatters={
            'Average Comm': '{:.4f}'.format,
            'Average Total MB': '{:.4f}'.format
        }
    ))

--- Bandwidth Metrics Summary ---
              Average Comm Average Total MB
Model                                      
attentioncomm      18.8317           5.0604
